# 03. Aggregate Incident Volume Forecasting & Evaluation
### Project: Smart Public Safety Analytics
**Focus:** Chronological time-series preparation, baseline naive forecasting, SARIMA modeling, lag-engineered Random Forest regression, and performance evaluation (MAE, RMSE, MAPE).

> **CRITICAL METHODOLOGICAL NOTICE:**
> Forecasts describe **aggregate historical statistical patterns** with uncertainty intervals. They do NOT predict individual behavior.


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np

import config
from src.data_loader import load_sample_dataset
from src.data_cleaning import clean_and_validate_data
from src.feature_engineering import engineer_features
from src.forecasting import (
    prepare_aggregate_series,
    run_chronological_evaluation,
    calculate_metrics
)


## 1. Prepare Aggregate Time Series


In [ ]:
raw_df, mapping, _, _ = load_sample_dataset()
cleaned_df, _ = clean_and_validate_data(raw_df, mapping)
featured_df = engineer_features(cleaned_df)

ts_daily = prepare_aggregate_series(featured_df, freq='D', area='All Areas', category='All Categories')
print(f"Daily Time Series: {len(ts_daily)} days ({ts_daily.index.min()} to {ts_daily.index.max()})")
print(f"Average Daily Incidents: {ts_daily.mean():.2f}")
ts_daily.head()


## 2. Chronological Train/Test Holdout Evaluation
Split into training history and subsequent test holdout periods (no random shuffling!).


In [ ]:
eval_results = run_chronological_evaluation(ts_daily, test_horizon=14)
print("Status:", eval_results['status'])

metrics_df = pd.DataFrame(eval_results['metrics']).T
print("\nChronological Model Evaluation Scorecard:")
metrics_df


## 3. Comparison of Actual vs Forecast


In [ ]:
test_s = eval_results['test']
preds = eval_results['predictions']

comparison_df = pd.DataFrame({
    'Actual': test_s,
    'SARIMA': preds['SARIMA'],
    'Random Forest (ML)': preds['Random Forest'],
    'Seasonal Naive': preds['Seasonal Naive']
})
comparison_df
